In [2]:
# importar librerias a usar
import pandas as pd
from clase_reportes_new import ReportClassNew
import numpy as np
import tkinter as tk
from tkinter import filedialog
import json
import re
import os
rc = ReportClassNew()

In [3]:
import pandas as pd
import holidays
from pathlib import Path
import os 
from thefuzz import process
from rapidfuzz import process
import os
import re
import tkinter as tk
from tkinter import filedialog
import numpy as np
import json
import unicodedata
from typing import Dict, Optional, List
from io import StringIO
import requests
from difflib import get_close_matches
import unicodedata
from collections import defaultdict
import fitz  # PyMuPDF



In [ ]:
dataset = rc.consolidar_carpeta(ruta_carpeta=r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA', extension='csv')
filtro =dataset[dataset['PRODUCTO'].isin(['[PCN20] PERFUME CAPILAR BOREAL','[PCN10] PERFUME CAPILAR PRAIA' ])]
filtro.to_excel(r'D:\Desktop\odoo_shopi.xlsx', index=False)

Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA
  - Archivo 'Ventas_Junio_2024_Diciembre_2024.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2025_Diciembre_2025.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2026_Marzo_2026.csv' leído correctamente.
  - Archivo 'Ventas_Abril_2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [5]:
# dataset = dataset[dataset['FECHA_FACTURA']>='2025-12-31']

In [6]:
import pandas as pd
import networkx as nx


# Limpiar datos clave
dataset['EMAIL'] = dataset['EMAIL'].str.lower().str.strip()
dataset['TELEFONO'] = dataset['TELEFONO'].str.replace(r'\D+', '', regex=True).str[-10:]
dataset['IDENTIFICACION_CLIENTE'] = dataset['IDENTIFICACION_CLIENTE'].astype(str).str.strip()
edges = []

for _, row in dataset.iterrows():
    valores = [
        row['EMAIL'],
        row['TELEFONO'],
        row['IDENTIFICACION_CLIENTE']
    ]
    
    valores = [v for v in valores if pd.notnull(v) and v != '']
    
    # conectar todos contra todos
    for i in range(len(valores)):
        for j in range(i+1, len(valores)):
            edges.append((valores[i], valores[j]))


G = nx.Graph()
G.add_edges_from(edges)
componentes = list(nx.connected_components(G))

mapa_cliente = {}

for i, comp in enumerate(componentes):
    for nodo in comp:
        mapa_cliente[nodo] = i
def obtener_id(row):
    for campo in ['EMAIL', 'TELEFONO', 'IDENTIFICACION_CLIENTE']:
        valor = row[campo]
        if valor in mapa_cliente:
            return mapa_cliente[valor]
    return None

dataset['CLIENTE_UNICO_ID'] = dataset.apply(obtener_id, axis=1)


In [11]:
dataset.to_csv(r'D:\Desktop\dataset_consolidado.csv', index=False, sep=';')

In [7]:
shopi = dataset[dataset['CATEGORÍA'] == 'SHOPIFY']

In [ ]:
sho

In [8]:
shopi['CATEGORÍA'].value_counts()

CATEGORÍA
SHOPIFY    71294
Name: count, dtype: int64

In [9]:
df_facturas = shopi[['CLIENTE_UNICO_ID', '﻿NUMERO_FACTURA']].drop_duplicates()

In [10]:
compras_por_cliente = (
    shopi
    .groupby('CLIENTE_UNICO_ID')['﻿NUMERO_FACTURA']
    .nunique()
    .reset_index(name='num_compras')
)

In [11]:
clientes_recompra = compras_por_cliente[compras_por_cliente['num_compras'] > 1]

In [12]:
compras_por_cliente

,CLIENTE_UNICO_ID,num_compras
0,0.0,2
1,1.0,2
2,2.0,1
3,3.0,3
4,4.0,3
...,...,...
27967,30690.0,1
27968,30691.0,1
27969,30692.0,1
27970,30693.0,1


In [13]:
tasa_recompra = len(clientes_recompra) / len(compras_por_cliente)
print(tasa_recompra)

0.08622908622908623


In [15]:
compras_por_cliente

,CLIENTE_UNICO_ID,num_compras
0,0.0,2
1,1.0,2
2,2.0,1
3,3.0,3
4,4.0,3
...,...,...
27967,30690.0,1
27968,30691.0,1
27969,30692.0,1
27970,30693.0,1


In [16]:
df_facturas


,CLIENTE_UNICO_ID,﻿NUMERO_FACTURA
133524,0.0,FEVY102135
133526,1.0,FEVY102136
133531,2.0,FEVY102137
133534,3.0,FEVY102138
133538,4.0,FEVY102139
...,...,...
503058,30691.0,FE30594
503060,30692.0,FE30595
503061,2431.0,FE30596
503062,30693.0,FE30597


In [14]:
dataset.to_csv(r"D:\Desktop\Come.csv", index=False, sep=';', encoding='utf-8-sig', delimiter=',')

TypeError: NDFrame.to_csv() got an unexpected keyword argument 'delimiter'

In [ ]:

# return  {'Base':df_resultado,
#     'nombre_archivo':nombre_archivo,
#     'facturas_afectadas':facturas_afectadas,
#     'errores':etiqueta_mayorista,
#     # 'asesores_sin_categoria':asesores_sin_categoria,
#     'cliente_call_center':cliente_call_center
#     }